# Episodic Memory

> **Store complete interaction episodes with temporal context, so the agent can recall "what happened when" across sessions.**

You don't remember your life as a flat list of facts. You remember *episodes*: bounded experiences tied to a time and place. "The meeting where we decided to pivot the product." "The debugging session that took all afternoon." Each episode has a beginning, a middle, and an end.

Episodic Memory brings this capability to AI agents. It captures entire conversation sessions (or meaningful segments) as discrete episodes. Each episode is tagged with timestamps, topic labels, and an auto-generated summary. This lets the agent recall past experiences as coherent events, not isolated facts.

This matters because many agent use cases span multiple sessions. A coaching agent needs to recall what goals were set last week. A project-management agent needs to know what was decided three days ago. Without episodic memory, each session starts from scratch. The agent has amnesia between interactions.

The key engineering challenge is **episode boundary detection** (deciding where one episode ends and another begins). Boundaries can be session-based (one session = one episode), topic-based (a topic shift starts a new episode), or time-based (a gap of *n* minutes triggers a new episode). A second challenge is retrieval: given a new user query, how do you find the most relevant past episodes? This requires a mix of temporal indexing, semantic similarity, and explicit reference detection.

**By the end of this notebook you'll understand:**
- How to capture conversation turns into structured episodes with metadata.
- How to detect episode boundaries (session-based and topic-based).
- How to generate episode summaries with an LLM.
- How to retrieve relevant episodes using recency and semantic similarity.
- When episodic memory helps and when it quietly fails.

## Key Concepts

- **Episode**: A bounded segment of conversation stored as a single unit. Think of it as one chapter in a journal. It has a start time, an end time, a list of messages, and a summary.
- **Episode boundary**: The dividing line between two episodes. Boundaries can be session-based (one session = one episode), topic-based (a detected topic shift starts a new episode), or time-based (a gap of *n* minutes triggers a new episode).
- **Temporal indexing**: Tagging each episode with timestamps (start time, end time, duration). This enables queries like "what did we discuss last Tuesday?" or "what happened in the previous session?"
- **Episode summary**: A short text that captures the gist of an episode. The LLM (large language model, the AI that generates text) produces this summary when the episode closes. Summaries let you scan past episodes quickly without loading full transcripts.
- **Embedding**: A list of numbers that represents the meaning of a piece of text. Two texts with similar meaning have embeddings that are close together in vector space. We use embeddings to find episodes that are semantically related to a query.
- **Cosine similarity**: A way to measure how close two embeddings are. It ranges from -1 (opposite) to 1 (identical). Higher values mean the texts are more related.
- **Episodic vs. semantic memory**: Episodic memory stores specific experiences ("the user asked me to rewrite the intro on March 5th"). Semantic memory stores generalized knowledge ("the user prefers concise writing"). Both are valuable but serve different roles.
- **Episode decay**: Older episodes may be compressed or pruned to keep storage manageable in long-lived agents.

## Architecture

<p align="center">
 <img src="../../images/diagrams/09_episodic_memory.svg" alt="Episodic Memory Architecture" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart LR
 Stream["Conversation\nStream"] --> Detector["Episode Boundary\nDetector"]
 Detector -- "episode complete" --> Packager["Episode Packager\n(metadata + summary)"]
 Packager --> Store["Episode Store\n(indexed by time, topic)"]
 Detector -- "continuing" --> Stream
 NewQuery["New User Query"] --> Retrieval["Retrieval Engine\n(temporal + semantic)"]
 Store --> Retrieval
 Retrieval --> Injection["Context Injection\n(episode summaries)"]
 Injection --> LLM["LLM"]
 LLM --> Response["Response"]
```

</details>

**Data flow:** The conversation stream flows through an episode boundary detector. When an episode is complete (session end, topic shift, or time gap), the episode packager creates a structured record. This record includes timestamps, topic tags, and an LLM-generated summary. The record goes into the episode store, indexed for both temporal and semantic retrieval. On a new query, the retrieval engine finds relevant past episodes and injects their summaries into the LLM prompt.

## Setup

Install dependencies and configure API access. You'll need an `OPENAI_API_KEY` environment variable set in a `.env` file.

In [ ]:
%pip install -q openai python-dotenv numpy

Import libraries and initialize the OpenAI client. The API key loads from a `.env` file.

In [ ]:
import os
import json
import uuid
import copy
from datetime import datetime, timedelta
from dataclasses import dataclass, field, asdict
from dotenv import load_dotenv
import numpy as np

import openai

load_dotenv() # reads OPENAI_API_KEY from .env

client = openai.OpenAI()

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

## Implementation

We'll build episodic memory in four steps:

1. Define the `Episode` data model.
2. Build an `EpisodicMemory` class with boundary detection, episode finalization, and retrieval.
3. Wire retrieved episodes into the LLM prompt.
4. Run a realistic multi-session example.

### Step 1: The Episode Data Model

Think of a filing cabinet where each folder holds one meeting's notes. The folder's label shows the date, the topic, and a one-line summary. You can grab the right folder without reading every page inside.

Our `Episode` dataclass works the same way. It stores the full transcript, but also carries metadata that makes searching fast.

In [ ]:
@dataclass
class Episode:
 """A single episode: a bounded segment of conversation with metadata."""

 episode_id: str
 start_time: str # ISO-format timestamp
 end_time: str # ISO-format timestamp
 messages: list[dict] # [{"role": ..., "content": ...}, ...]
 summary: str # LLM-generated summary
 topic_tags: list[str] # extracted topic labels
 summary_embedding: list[float] = field(default_factory=list) # for semantic search

 def to_dict(self) -> dict:
 return asdict(self)

 @classmethod
 def from_dict(cls, data: dict) -> "Episode":
 return cls(**data)

 def __repr__(self) -> str:
 tag_str = ", ".join(self.topic_tags[:3])
 return (
 f"Episode({self.episode_id[:8]}... | "
 f"{len(self.messages)} msgs | "
 f"tags=[{tag_str}])"
 )

### Step 2: Helper Functions

We need three helper functions that call the LLM or the embedding API:

1. **Summarize** an episode's messages into a short paragraph.
2. **Extract topic tags** from the messages.
3. **Get an embedding** for a piece of text (used for semantic search).

In [ ]:
def summarize_messages(messages: list[dict]) -> str:
 """Ask the LLM to produce a concise summary of a conversation segment."""
 transcript = "\n".join(
 f"{m['role'].upper()}: {m['content']}" for m in messages
 )
 response = client.chat.completions.create(
 model=MODEL,
 messages=[
 {
 "role": "system",
 "content": (
 "Summarize the following conversation in 2-3 sentences. "
 "Focus on key decisions, facts shared, and outcomes."
 ),
 },
 {"role": "user", "content": transcript},
 ],
 max_tokens=200,
 )
 return response.choices[0].message.content.strip()


def extract_topic_tags(messages: list[dict]) -> list[str]:
 """Ask the LLM to extract 2-4 topic labels from a conversation segment."""
 transcript = "\n".join(
 f"{m['role'].upper()}: {m['content']}" for m in messages
 )
 response = client.chat.completions.create(
 model=MODEL,
 messages=[
 {
 "role": "system",
 "content": (
 "Extract 2-4 short topic tags from the conversation below. "
 "Return them as a JSON array of lowercase strings. "
 "Example: [\"budgeting\", \"q3 goals\", \"team hiring\"]"
 ),
 },
 {"role": "user", "content": transcript},
 ],
 max_tokens=100,
 )
 raw = response.choices[0].message.content.strip()
 try:
 tags = json.loads(raw)
 if isinstance(tags, list):
 return [str(t).lower() for t in tags]
 except json.JSONDecodeError:
 pass
 # Fallback: split by comma if JSON parsing fails
 return [t.strip().lower().strip('"') for t in raw.split(",")][:4]




We also need functions for embedding and similarity. `get_embedding` converts text into a vector (a list of numbers that captures meaning). `cosine_similarity` measures how close two vectors are. We use these to find past episodes that are semantically related to a new query.

In [ ]:
def get_embedding(text: str) -> list[float]:
 """Get a vector embedding for a piece of text."""
 response = client.embeddings.create(
 model=EMBEDDING_MODEL,
 input=text,
 )
 return response.data[0].embedding


def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
 """Compute cosine similarity between two vectors."""
 a = np.array(vec_a)
 b = np.array(vec_b)
 dot = np.dot(a, b)
 norm = np.linalg.norm(a) * np.linalg.norm(b)
 if norm == 0:
 return 0.0
 return float(dot / norm)

### Step 3: The EpisodicMemory Class

This is the core class. It accumulates messages into a buffer, detects when an episode is complete, finalizes the episode (summary + tags + embedding), and stores it. It also retrieves relevant past episodes when the agent needs context.

We support two boundary strategies:
- **Session-based**: you call `end_session()` explicitly to close an episode.
- **Topic-based**: the LLM checks whether the latest message shifted the topic.

In [ ]:
class EpisodicMemory:
 """Manages episodic memory: accumulation, boundary detection, and retrieval."""

 def __init__(self, boundary_strategy: str = "session"):
 """
 Args:
 boundary_strategy: "session" (manual) or "topic" (LLM-detected).
 """
 self.episodes: list[Episode] = []
 self.current_buffer: list[dict] = []
 self.current_start_time: str = datetime.now().isoformat()
 self.boundary_strategy = boundary_strategy

 # ── Adding messages ──────────────────────────────────────────

 def add_message(self, role: str, content: str) -> bool:
 """Add a message to the current buffer. Returns True if an episode was finalized."""
 self.current_buffer.append({"role": role, "content": content})

 if self.boundary_strategy == "topic" and len(self.current_buffer) >= 4:
 if self._detect_topic_shift():
 # The latest message belongs to a new topic.
 # Finalize everything *before* it.
 new_topic_msg = self.current_buffer.pop()
 self._finalize_episode()
 # Start the new episode with the shifted message
 self.current_buffer = [new_topic_msg]
 self.current_start_time = datetime.now().isoformat()
 return True
 return False



Next we add boundary detection. The `_detect_topic_shift` method asks the LLM whether the latest message changed the subject. If yes, the current buffer is finalized as an episode and a new one begins.

In [ ]:
 # ── Boundary detection ───────────────────────────────────────

 def _detect_topic_shift(self) -> bool:
 """Ask the LLM whether the last message shifted the topic."""
 recent = self.current_buffer[-4:] # last 4 messages for context
 transcript = "\n".join(
 f"{m['role'].upper()}: {m['content']}" for m in recent
 )
 response = client.chat.completions.create(
 model=MODEL,
 messages=[
 {
 "role": "system",
 "content": (
 "You are a conversation analyst. Look at the last few messages below. "
 "Did the LAST message introduce a significantly different topic "
 "from what was being discussed before it? "
 "Answer with exactly one word: YES or NO."
 ),
 },
 {"role": "user", "content": transcript},
 ],
 max_tokens=5,
 )
 answer = response.choices[0].message.content.strip().upper()
 return answer.startswith("YES")



The `end_session` method is the manual boundary for session-based mode. `_finalize_episode` packages the current buffer into a complete Episode: it generates a summary, extracts topic tags, computes an embedding, and resets the buffer.

In [ ]:
 # ── Session-based boundary ───────────────────────────────────

 def end_session(self) -> Episode | None:
 """Manually close the current episode (session-based boundary)."""
 if not self.current_buffer:
 return None
 return self._finalize_episode()

 # ── Episode finalization ─────────────────────────────────────

 def _finalize_episode(self) -> Episode:
 """Package the current buffer into a complete Episode."""
 summary = summarize_messages(self.current_buffer)
 tags = extract_topic_tags(self.current_buffer)
 embedding = get_embedding(summary)

 episode = Episode(
 episode_id=str(uuid.uuid4()),
 start_time=self.current_start_time,
 end_time=datetime.now().isoformat(),
 messages=copy.deepcopy(self.current_buffer),
 summary=summary,
 topic_tags=tags,
 summary_embedding=embedding,
 )
 self.episodes.append(episode)

 # Reset the buffer
 self.current_buffer = []
 self.current_start_time = datetime.now().isoformat()

 return episode



The `retrieve` method finds the most relevant past episodes for a given query. It scores each episode by combining semantic similarity (how close the query is to the episode's summary) with recency (newer episodes score higher). You control the balance with `recency_weight` and `semantic_weight`.

In [ ]:
 # ── Retrieval ────────────────────────────────────────────────

 def retrieve(
 self,
 query: str,
 k: int = 3,
 recency_weight: float = 0.3,
 semantic_weight: float = 0.7,
 ) -> list[tuple[Episode, float]]:
 """
 Retrieve the top-k most relevant episodes.

 Scoring combines:
 - Semantic similarity (cosine between query embedding and episode summary embedding).
 - Recency (more recent episodes score higher).

 Returns a list of (episode, score) tuples, sorted by score descending.
 """
 if not self.episodes:
 return []

 query_embedding = get_embedding(query)

 # Compute recency scores: the most recent episode gets 1.0, the oldest gets 0.0
 n = len(self.episodes)
 recency_scores = [i / max(n - 1, 1) for i in range(n)]

 scored = []
 for idx, episode in enumerate(self.episodes):
 sem_score = cosine_similarity(query_embedding, episode.summary_embedding)
 rec_score = recency_scores[idx]
 combined = (semantic_weight * sem_score) + (recency_weight * rec_score)
 scored.append((episode, combined))

 scored.sort(key=lambda x: x[1], reverse=True)
 return scored[:k]

 # ── Utilities ────────────────────────────────────────────────

 def get_all_episodes(self) -> list[Episode]:
 return list(self.episodes)

 def __len__(self) -> int:
 return len(self.episodes)

 def __repr__(self) -> str:
 return f"EpisodicMemory({len(self.episodes)} episodes, strategy={self.boundary_strategy})"

### Step 4: Wiring Episodes into the LLM Prompt

When the agent receives a new query, we retrieve relevant past episodes and inject their summaries into the system prompt. This gives the LLM awareness of what happened in previous sessions.

In [ ]:
def build_system_prompt(base_prompt: str, episodes: list[tuple]) -> str:
 """Build a system prompt that includes retrieved episode summaries."""
 if not episodes:
 return base_prompt

 episode_context = "\n\n".join(
 f"--- Episode from {ep.start_time[:16]} (topics: {', '.join(ep.topic_tags)}) ---\n"
 f"{ep.summary}"
 for ep, _score in episodes
 )

 return (
 f"{base_prompt}\n\n"
 f"## Relevant past episodes\n\n"
 f"{episode_context}"
 )


def chat_with_episodic_memory(
 memory: EpisodicMemory,
 user_input: str,
 session_messages: list[dict],
 base_system_prompt: str = "You are a helpful assistant with access to past conversation episodes.",
) -> str:
 """Send a message to the LLM with episode context injected."""
 # Retrieve relevant past episodes
 relevant_episodes = memory.retrieve(user_input, k=3)

 # Build system prompt with episode summaries
 system_prompt = build_system_prompt(base_system_prompt, relevant_episodes)

 # Add the new user message to the session
 session_messages.append({"role": "user", "content": user_input})

 response = client.chat.completions.create(
 model=MODEL,
 messages=[
 {"role": "system", "content": system_prompt},
 *session_messages,
 ],
 max_tokens=512,
 )

 assistant_text = response.choices[0].message.content.strip()
 session_messages.append({"role": "assistant", "content": assistant_text})

 # Record messages in the episodic memory buffer
 memory.add_message("user", user_input)
 memory.add_message("assistant", assistant_text)

 return assistant_text

## Example Run

Let's simulate a realistic scenario: a project-coaching agent that talks to a user across three sessions over several days. After each session, we finalize the episode. Then in the final session, we test whether the agent can recall past episodes.

### Session 1: Project Kickoff

In [ ]:
memory = EpisodicMemory(boundary_strategy="session")

# --- Session 1: Project kickoff ---
session1_messages = []
base_prompt = (
 "You are a project-coaching assistant. Keep replies concise (2-3 sentences). "
 "Reference past episodes when relevant."
)

exchanges_s1 = [
 "Hi! I'm starting a new project. We're building a recommendation engine for an e-commerce platform.",
 "The team has 4 engineers. We're using Python and PostgreSQL. Timeline is 3 months.",
 "For the first milestone, we want a basic collaborative filtering model ready in 3 weeks.",
]

print("=" * 60)
print("SESSION 1: Project Kickoff")
print("=" * 60)

for msg in exchanges_s1:
 print(f"\n User: {msg}")
 reply = chat_with_episodic_memory(memory, msg, session1_messages, base_prompt)
 print(f" Agent: {reply}")

# Finalize session 1
ep1 = memory.end_session()
print(f"\n [Episode finalized] {ep1}")
print(f" Summary: {ep1.summary}")
print(f" Tags: {ep1.topic_tags}")

### Session 2: Progress Check

A few days later, the user returns to discuss progress. Notice that `session2_messages` starts empty (a fresh session). The agent recovers context from the stored episode.

In [ ]:
# --- Session 2: Progress check ---
session2_messages = [] # fresh session, no message history

exchanges_s2 = [
 "Hey, quick update. The collaborative filtering prototype is working. We hit 78% precision on the test set.",
 "But we're running into cold-start issues with new users. Any suggestions?",
 "Good idea. We'll try content-based fallback for new users with fewer than 5 interactions.",
]

print("=" * 60)
print("SESSION 2: Progress Check")
print("=" * 60)

for msg in exchanges_s2:
 print(f"\n User: {msg}")
 reply = chat_with_episodic_memory(memory, msg, session2_messages, base_prompt)
 print(f" Agent: {reply}")

# Finalize session 2
ep2 = memory.end_session()
print(f"\n [Episode finalized] {ep2}")
print(f" Summary: {ep2.summary}")
print(f" Tags: {ep2.topic_tags}")

### Session 3: Recall Test

Now the user asks the agent to recall what happened in past sessions. The agent has no conversation history for this session. It depends entirely on episodic memory.

In [ ]:
# --- Session 3: Recall test ---
session3_messages = [] # fresh session again

recall_questions = [
 "What project am I working on and what's the team size?",
 "What was the precision score we achieved on the collaborative filtering model?",
 "What problem did we identify with new users, and what was the proposed solution?",
]

print("=" * 60)
print("SESSION 3: Recall Test")
print("=" * 60)

for msg in recall_questions:
 print(f"\n User: {msg}")
 reply = chat_with_episodic_memory(memory, msg, session3_messages, base_prompt)
 print(f" Agent: {reply}")

print("\n" + "=" * 60)
print(f"Total episodes stored: {len(memory)}")
print("=" * 60)

### Inspecting the Episode Store

Let's look at what's stored. Each episode has a time range, tags, and a summary. This is what the retrieval engine searches through.

In [ ]:
print("All stored episodes:\n")
for i, ep in enumerate(memory.get_all_episodes()):
 print(f"Episode {i + 1}:")
 print(f" ID: {ep.episode_id[:12]}...")
 print(f" Time: {ep.start_time[:16]} to {ep.end_time[:16]}")
 print(f" Messages: {len(ep.messages)}")
 print(f" Tags: {ep.topic_tags}")
 print(f" Summary: {ep.summary}")
 print()

### Testing Retrieval Directly

Let's see how the retrieval engine scores episodes for different queries. A query about "cold-start" should rank Session 2 higher. A query about "project setup" should rank Session 1 higher.

In [ ]:
test_queries = [
 "What was the cold-start problem we discussed?",
 "Tell me about the project setup and team.",
 "What technology stack are we using?",
]

for query in test_queries:
 print(f"Query: \"{query}\"")
 results = memory.retrieve(query, k=3)
 for ep, score in results:
 print(f" Score {score:.3f} | tags={ep.topic_tags} | {ep.summary[:80]}...")
 print()

## Persistence

A production episodic memory must survive process restarts. Since each `Episode` is a dataclass, we can serialize the entire store to JSON.

In [ ]:
def save_episodes(memory: EpisodicMemory, path: str) -> None:
 """Save all episodes to a JSON file."""
 data = [ep.to_dict() for ep in memory.episodes]
 with open(path, "w") as f:
 json.dump(data, f, indent=2)
 print(f"Saved {len(data)} episodes to {path}")


def load_episodes(path: str, boundary_strategy: str = "session") -> EpisodicMemory:
 """Load episodes from a JSON file into a new EpisodicMemory."""
 with open(path) as f:
 data = json.load(f)
 mem = EpisodicMemory(boundary_strategy=boundary_strategy)
 mem.episodes = [Episode.from_dict(d) for d in data]
 print(f"Loaded {len(mem.episodes)} episodes from {path}")
 return mem


# Demo round-trip
save_episodes(memory, "episodes.json")
loaded_memory = load_episodes("episodes.json")

# Verify the loaded memory works
results = loaded_memory.retrieve("What is the project about?", k=2)
for ep, score in results:
 print(f" Retrieved: score={score:.3f} | {ep.summary[:80]}...")

## Tradeoffs

### When Episodic Memory Works Well

- **Multi-session agents** where users expect the agent to remember what happened last time. A coaching agent, a project tracker, or a recurring meeting assistant all benefit.
- **Temporal reasoning**: when the user asks "what did we decide last week?" or "how has this changed over time?", episode timestamps make these queries possible.
- **Accountability and auditing**: each episode is a complete record. You can review what the agent knew and when it knew it.

### When It Breaks Down

- **Boundary detection is hard**. Session-based boundaries are straightforward but coarse (one long session becomes one giant episode). Topic-based detection requires extra LLM calls and can misfire.
- **Storage grows with every session**. A long-lived agent accumulates hundreds of episodes. Without pruning or archival, retrieval slows down and costs increase.
- **Summaries lose detail**. The summary captures the gist, but fine-grained facts may be lost. If the user asks about a specific number from three sessions ago, the summary might not include it.
- **Retrieval adds latency**. Every user message triggers an embedding call plus a scan of all episodes. For agents with thousands of episodes, you'll need a vector database instead of a linear scan.

### Comparison with Other Memory Patterns

| Aspect | Buffer Memory | Summary Memory | Episodic Memory |
|--------|--------------|----------------|----------------|
| **Scope** | Single session | Single session | Cross-session |
| **Granularity** | Every message | Compressed summary | Complete episodes |
| **Temporal awareness** | Turn order only | None | Full timestamps |
| **Retrieval** | Send everything | Send summary | Search by relevance |
| **Best for** | Short conversations | Long single sessions | Multi-session agents |

## Further Reading

- [Tulving, "Episodic Memory: From Mind to Brain," *Annual Review of Psychology*, 2002](https://doi.org/10.1146/annurev.psych.53.100901.135114): The foundational paper that defined episodic memory and distinguished it from semantic memory.
- [Park et al., "Generative Agents: Interactive Simulacra of Human Behavior," UIST 2023](https://arxiv.org/abs/2304.03442): Shows agents with episodic memory that remember and reflect on past experiences in a simulated town.
- [Letta (MemGPT)](https://github.com/cpacker/MemGPT): An open-source implementation of tiered memory management, including episodic recall for LLM agents.
- [Conway, "Episodic Memories," *Neuropsychologia*, 2009](https://doi.org/10.1016/j.neuropsychologia.2009.02.003): A cognitive science view of how episodic memories are structured and retrieved in the human brain.
- [OpenAI Embeddings Guide](https://platform.openai.com/docs/guides/embeddings?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Reference for the embedding API used in this notebook's semantic retrieval.

---

*\u2190 Previous: [08 - Knowledge Graph Memory](../08_knowledge_graph_memory/) \u00b7 Next: [10 - Semantic Memory](../10_semantic_memory/) \u2192*

Clean up temporary files created during the demo.

In [ ]:
for f in ["episodes.json"]:
 if os.path.exists(f):
 os.remove(f)
 print(f"Removed {f}")

## 🧪 Try It Yourself

Three small challenges to deepen your understanding. Each should take 10-30 minutes.

### Challenge 1: Embedding-based boundary detection
Replace `_detect_topic_shift()` with a version that computes the cosine distance between the embedding of the latest message and the mean embedding of the current episode buffer. Trigger a new episode when the distance exceeds a threshold. Compare detection accuracy against the LLM-based approach.

### Challenge 2: Retrieval weight tuning
The `retrieve()` method blends semantic similarity and recency. Vary the recency weight from 0.0 to 1.0 in steps of 0.2. For each setting, run 5 retrieval queries and measure which episodes are returned. Plot the overlap between settings to see how sensitive retrieval is to the weight.

### Challenge 3: Cross-session episode linking
After ending a session with `end_session()`, start a new session. Use `save_episodes()` and `load_episodes()` to persist episodes. In the new session, retrieve relevant episodes from the previous session and use them as context. This directly combines episodic memory with the persistence strategy from 21 Cross-Session Memory.


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--09-episodic-memory--episodic-memory)
